# KV-cache eviction and a planted complaint: Colab T4 runbook

**Runtime → Change runtime type → T4 GPU**, then run top to bottom.

Before you start, upload the repo to `MyDrive/kv-complaints`, **with** `results/manifest.json`
and `data/review_texts.json` from your laptop build and **without** the `yelp/` folder.
Results are written to Drive, so a disconnect loses nothing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/kv-complaints
!ls results/manifest.json data/review_texts.json

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Mounted at /content/drive
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_13294/3406883473.py", line 3, in <cell line: 0>
    get_ipython().run_line_magic('cd', '/content/drive/MyDrive/kv-complaints')
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
OSError: [Errn

In [15]:
!pip install -q "kvpress==0.5.5" "transformers==4.57.*" accelerate pytest
!pip freeze > requirements.lock
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "fire>=0.7"

name, memory.total [MiB]
Tesla T4, 15360 MiB
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kvpress 0.5.5 requires fire<0.7,>=0.6.0, but you have fire 0.7.1 which is incompatible.


If an import fails after pip changed numpy or pandas: **Runtime → Restart session**, re-run the first cell, continue here.

## Git
`run.py` records the commit of every run, and Test 4 checks every committed file. Edit your name and email once.

In [16]:
!git config --global --add safe.directory '*'
!git config --global user.name "PranavaMR"
!git config --global user.email "aragornoftirith@gmail.com"
!test -d .git || git init -q
!git add -A && git commit -qm "Freeze code, lexicon and templates (code drafted with Claude)" || true
!git log --oneline | head -3

254a81f Freeze code, lexicon and templates (code drafted with Claude)
85948c8 Freeze code, lexicon and templates (code drafted with Claude)


In [11]:
!pip install pipes

ERROR: Could not find a version that satisfies the requirement pipes (from versions: none)
ERROR: No matching distribution found for pipes


## Smoke test: KnormPress at 50% on a ~2k-token dummy context

In [17]:

import sys, shlex
sys.modules['pipes'] = shlex

import torch, kvpress
from kvpress import KnormPress
from run import load_pipe

pipe = load_pipe()
dummy = ' '.join(['The soup was warm and the staff were kind.'] * 200)
print(pipe(dummy, question='\n\nWhat did people say about the soup?',
           press=KnormPress(compression_ratio=0.5), max_new_tokens=40)['answer'])
print('model revision:', pipe.model.config._commit_hash)
del pipe; torch.cuda.empty_cache()

Device set to use cuda:0


People said the soup was warm and the staff were kind.
model revision: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306


## Tests 1 to 4
`-rs` lists skipped tests: all four must say passed, none skipped.

In [18]:
!python -m pytest -q -rs tests

.......                                                                  [100%]
7 passed in 47.99s


## Calibration
Full cache, 12 planted contexts, Q1 only. If Full mentions the complaint in under 30% or over 95%,
change `N_RECOMMENDATIONS` in `config.py` **once** (to 8 or 3), log it in `logs/decisions.md`, commit, then launch.

In [19]:
!python run.py --calibrate

I0000 00:00:1794334500.818458    9642 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Device set to use cuda:0
Traceback (most recent call last):
  File "/content/drive/MyDrive/kv-complaints/run.py", line 156, in <module>
    calibrate()
    ~~~~~~~~~^^
  File "/content/drive/MyDrive/kv-complaints/run.py", line 144, in calibrate
    run(contexts, ["full"], config.CALIBRATION_FILE, ask_q2=False)
    ~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/kv-complaints/run.py", line 104, in run
    assert torch.equal(expected, rendered.context_ids), "render() disagrees with kvpress preprocessing"
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: render() disagrees with kvpress preprocessing


## The 240 runs
Roughly 1 to 1.5 h on a T4. If Colab disconnects, re-run the first cell and this one: it resumes.

In [ ]:
!python run.py

In [ ]:
!wc -l data/runs_raw.jsonl

## Redact, label, analyse
1. `redact.py` writes the committable `results/runs.jsonl`.
2. `labels.py` writes `labels/to_label.csv`. Label every row (1 per aspect the advice addresses, else 0)
   and save it as `labels/labels.csv`. Don't open `labels/key.csv`.
3. Run the analysis, pick the primary detector by your pre-registered rule, set `PRIMARY_DETECTOR`
   in `config.py`, and run the analysis again.

In [ ]:
!python redact.py
!python labels.py

In [ ]:
!python analysis.py

In [ ]:
from IPython.display import Image, display
display(Image('results/figures/fig1_mention_rate.png'))
display(Image('results/figures/fig2_retention.png'))

## Final commit
First clear this notebook's outputs (**Edit → Clear all outputs**, then **File → Save**), so no answer text is saved in it.
Files are staged before the tests, so Test 4 checks everything that is about to be committed.

In [ ]:
!git add -A
!python -m pytest -q -rs tests && git commit -qm "Runs, labels, tables and figures" && git tag v-assessment
!git log --oneline | head -3